## Practing EDA and data cleaning

In [12]:
from configs.settings import project_dir
import pandas as pd
import numpy as np 

In [13]:
dir_p = project_dir()
data = dir_p.RAW_DATA_DIR/'googleplaystore.csv'
df = pd.read_csv(data)
df.head()

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19M,"10,000+",Free,0,Everyone,Art & Design,"January 7, 2018",1.0.0,4.0.3 and up
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7M,"5,000,000+",Free,0,Everyone,Art & Design,"August 1, 2018",1.2.4,4.0.3 and up
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25M,"50,000,000+",Free,0,Teen,Art & Design,"June 8, 2018",Varies with device,4.2 and up
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2.8M,"100,000+",Free,0,Everyone,Art & Design;Creativity,"June 20, 2018",1.1,4.4 and up


#### Looking into the data

In [14]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 10841 entries, 0 to 10840
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   App             10841 non-null  str    
 1   Category        10841 non-null  str    
 2   Rating          9367 non-null   float64
 3   Reviews         10841 non-null  str    
 4   Size            10841 non-null  str    
 5   Installs        10841 non-null  str    
 6   Type            10840 non-null  str    
 7   Price           10841 non-null  str    
 8   Content Rating  10840 non-null  str    
 9   Genres          10841 non-null  str    
 10  Last Updated    10841 non-null  str    
 11  Current Ver     10833 non-null  str    
 12  Android Ver     10838 non-null  str    
dtypes: float64(1), str(12)
memory usage: 1.1 MB


The homework for the participants is to replace the missing values in the 'Android version' and 'current version' columns intelligently. Additionally, they are tasked with replacing the missing 'Size' values with the appropriate mean or median, considering category-wise or genre-wise means instead of just a global median.

add handling the last updated date too.

In [15]:
# missing values
df.isnull().sum()

App                  0
Category             0
Rating            1474
Reviews              0
Size                 0
Installs             0
Type                 1
Price                0
Content Rating       1
Genres               0
Last Updated         0
Current Ver          8
Android Ver          3
dtype: int64

In [16]:
# changing the datatype of last updated columns
df['Last Updated'].head(2)

0     January 7, 2018
1    January 15, 2018
Name: Last Updated, dtype: str

In [17]:
df['Last Updated'].str.endswith('19').sum()

np.int64(1)

In [18]:
df['Last Updated'].dtype
df['Last Updated'] = pd.to_datetime(df['Last Updated'],errors='coerce')

In [19]:
df['Last Updated'].value_counts()

Last Updated
2018-08-03    326
2018-08-02    304
2018-07-31    294
2018-08-01    285
2018-07-30    211
             ... 
2014-11-25      1
2016-05-19      1
2014-01-20      1
2014-02-16      1
2014-03-23      1
Name: count, Length: 1377, dtype: int64

In [20]:
df = df[df['Type'] != '0']
df['Type'].value_counts()

Type
Free    10039
Paid      800
Name: count, dtype: int64

In [21]:
df['Price'].value_counts()

Price
0          10040
$0.99        148
$2.99        129
$1.99         73
$4.99         72
           ...  
$3.61          1
$394.99        1
$1.26          1
$1.20          1
$1.04          1
Name: count, Length: 92, dtype: int64

In [26]:
# handling the missing value in current ver and android ver columns
df[['Current Ver','Android Ver']].head()
df.head()
df.groupby('Android Ver')['Current Ver'].value_counts()

Android Ver         Current Ver
1.0 and up          0.7            1
                    2.0            1
1.5 and up          1.0            5
                    2.0            1
                    1.0.28         1
                                  ..
Varies with device  4.2.11         1
                    1.078          1
                    1.6            1
                    1.0.2          1
                    6.3.2          1
Name: count, Length: 4713, dtype: int64

In [ ]:
df.groupby('Last Updated')['Android '].value_counts().to_string('last_updated_and_android_ver.txt')


In [ ]:
df.tail(20)

In [43]:
df['Android Ver'] = df['Android Ver'].fillna('mode')
df['Android_ver_clean'] = df['Android Ver'].str.replace('and up','')

In [44]:
df.isnull().sum()

App                     0
Category                0
Rating               1474
Reviews                 0
Size                    0
Installs                0
Type                    1
Price                   0
Content Rating          0
Genres                  0
Last Updated            0
Current Ver             8
Android Ver             0
Android_ver_clean       0
dtype: int64

In [46]:
df.groupby('Android_ver_clean')['Current Ver'].value_counts().to_string('Android_Ver_and_Current_ver.txt')

In [57]:
# replacin the unwanted str with text
df['Current_ver_clean'] = np.where(
    df['Current Ver'].str.replace(' ','').str.isalpha(), "Text", df['Current Ver']
)

# handling the value that contains letters
df['Current_ver_clean'] = df['Current_ver_clean'].str.extract(r"(\d+(?:\.\d+))+")
df['Current_ver_clean'].head(20)
df.groupby('Android_ver_clean')['Current_ver_clean'].value_counts().to_string('clean_current_ver.txt')

In [59]:
# handlilng nan values in Current_ver_clean col
df['Current_ver_clean'] = df['Current_ver_clean'].fillna('mode')

In [60]:
df.isnull().sum()

App                     0
Category                0
Rating               1474
Reviews                 0
Size                    0
Installs                0
Type                    1
Price                   0
Content Rating          0
Genres                  0
Last Updated            0
Current Ver             8
Android Ver             0
Android_ver_clean       0
Current_ver_clean       0
dtype: int64

In [48]:
data = '34'
data.isalpha()

False

In [58]:
df.head(20)

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver,Android_ver_clean,Current_ver_clean
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19M,"10,000+",Free,0,Everyone,Art & Design,2018-01-07,1.0.0,4.0.3 and up,4.0.3,1.0
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,2018-01-15,2.0.0,4.0.3 and up,4.0.3,2.0
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7M,"5,000,000+",Free,0,Everyone,Art & Design,2018-08-01,1.2.4,4.0.3 and up,4.0.3,1.2
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25M,"50,000,000+",Free,0,Teen,Art & Design,2018-06-08,Varies with device,4.2 and up,4.2,NaN
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2.8M,"100,000+",Free,0,Everyone,Art & Design;Creativity,2018-06-20,1.1,4.4 and up,4.4,1.1
5,Paper flowers instructions,ART_AND_DESIGN,4.4,167,5.6M,"50,000+",Free,0,Everyone,Art & Design,2017-03-26,1.0,2.3 and up,2.3,1.0
6,Smoke Effect Photo Maker - Smoke Editor,ART_AND_DESIGN,3.8,178,19M,"50,000+",Free,0,Everyone,Art & Design,2018-04-26,1.1,4.0.3 and up,4.0.3,1.1
7,Infinite Painter,ART_AND_DESIGN,4.1,36815,29M,"1,000,000+",Free,0,Everyone,Art & Design,2018-06-14,6.1.61.1,4.2 and up,4.2,6.1
8,Garden Coloring Book,ART_AND_DESIGN,4.4,13791,33M,"1,000,000+",Free,0,Everyone,Art & Design,2017-09-20,2.9.2,3.0 and up,3.0,2.9
9,Kids Paint Free - Drawing Fun,ART_AND_DESIGN,4.7,121,3.1M,"10,000+",Free,0,Everyone,Art & Design;Creativity,2018-07-03,2.8,4.0.3 and up,4.0.3,2.8
